<a href="https://colab.research.google.com/github/hodyek/lung-colon-cancer-histopathology/blob/main/Notebooks/04_efficientnet_b0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 04: Transfer Learning — EfficientNet-B0

## Overview
This notebook fine-tunes EfficientNet-B0 pretrained on ImageNet for five-class classification on the LC25000 dataset. EfficientNet-B0 was chosen because it achieves strong accuracy with a compact architecture, making it well suited for deployment. We use a two-phase training strategy: first training only the classifier head, then unfreezing the full network for fine-tuning.

## Objectives
1. Load the same train/val/test splits used in Notebook 03.
2. Build EfficientNet-B0 with a replaced classifier head.
3. Train in two phases: head-only warmup, then full fine-tuning.
4. Plot training and validation curves.
5. Evaluate on the test set with all metrics.
6. Plot confusion matrix and ROC curves.
7. Document the overfitting gap and compare to the baseline.

In [ ]:
!pip install torch torchvision --quiet

import os, random, warnings, json, copy, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from google.colab import drive

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
drive.mount('/content/drive', force_remount=False)

BASE_DIR    = Path('/content/drive/MyDrive/lung-colon-cancer-histopathology')
FIGURES_DIR = BASE_DIR / 'figures'
MODELS_DIR  = BASE_DIR / 'models'

with open(BASE_DIR / 'data' / 'dataset_splits.json', 'r') as f:
    split_data = json.load(f)

train_paths  = split_data['train_paths']
val_paths    = split_data['val_paths']
test_paths   = split_data['test_paths']
train_labels = split_data['train_labels']
val_labels   = split_data['val_labels']
test_labels  = split_data['test_labels']
CLASS_NAMES  = split_data['class_names']

print(f'Train: {len(train_paths):,} | Val: {len(val_paths):,} | Test: {len(test_paths):,}')

In [ ]:
# DataLoaders — identical to Notebook 03 for fair comparison

IMAGE_SIZE    = 224
BATCH_SIZE    = 32
NUM_WORKERS   = 2
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class LC25000Dataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_loader = DataLoader(LC25000Dataset(train_paths, train_labels, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(LC25000Dataset(val_paths,   val_labels,   eval_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(LC25000Dataset(test_paths,  test_labels,  eval_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print('DataLoaders ready.')

In [ ]:
# ── Build EfficientNet-B0 ─────────────────────────────────────────────────────

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Freeze all backbone layers for warmup phase
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier head
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, 5),
)

model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'EfficientNet-B0 loaded with ImageNet weights.')
print(f'Trainable parameters (warmup): {trainable:,} / {total:,}')

In [ ]:
# ── Phase 1: Warmup — train head only for 5 epochs ───────────────────────────

criterion  = nn.CrossEntropyLoss()
optimizer  = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
CHECKPOINT = str(MODELS_DIR / 'efficientnet_b0_best.pth')

def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += outputs.argmax(1).eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, correct / total

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print('Phase 1: Warmup (head only, 5 epochs)')
for epoch in range(1, 6):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    print(f'  Epoch {epoch}/5 | Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}')

print('Warmup complete.')

In [ ]:
# ── Phase 2: Full fine-tuning — unfreeze all layers ───────────────────────────

for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'All layers unfrozen. Trainable parameters: {trainable:,}')

optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

NUM_EPOCHS        = 25
PATIENCE          = 7
best_val_loss     = float('inf')
best_weights      = copy.deepcopy(model.state_dict())
epochs_no_improve = 0

print('\nPhase 2: Full fine-tuning')
for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion)
    scheduler.step(vl_loss)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    elapsed = time.time() - start
    print(f'  Epoch {epoch:>3}/{NUM_EPOCHS} | '
          f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | '
          f'Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f} | '
          f'Time: {elapsed:.1f}s')

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_weights  = copy.deepcopy(model.state_dict())
        torch.save(best_weights, CHECKPOINT)
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}.')
            break

model.load_state_dict(best_weights)
print(f'\nBest val loss: {best_val_loss:.4f}')
print(f'Model saved to: {CHECKPOINT}')

In [ ]:
# Training curves
epochs_ran = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('EfficientNet-B0 — Training Curves', fontsize=14)

axes[0].plot(epochs_ran, history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(epochs_ran, history['val_loss'],   label='Val Loss',   linewidth=2)
axes[0].axvline(x=5, color='gray', linestyle='--', linewidth=1, label='Warmup end')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, history['train_acc'], label='Train Acc', linewidth=2)
axes[1].plot(epochs_ran, history['val_acc'],   label='Val Acc',   linewidth=2)
axes[1].axvline(x=5, color='gray', linestyle='--', linewidth=1, label='Warmup end')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
save_path = FIGURES_DIR / '04_efficientnet_training_curves.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {save_path}')

### Observation — Training Curves

The two-phase training strategy shows clearly in the curves. During the warmup phase the backbone is frozen so only the new classifier head trains, and both loss and accuracy change quickly. When the full network is unfrozen in Phase 2, the learning rate is lowered to 1e-4 to avoid destroying the pretrained feature representations. The vertical dashed line at epoch 5 marks this transition. Transfer learning from ImageNet weights gives EfficientNet-B0 a strong starting point, so it typically reaches a useful accuracy much faster than the baseline CNN trained from scratch.

In [ ]:
# Test evaluation
model.eval()
all_labels, all_preds, all_probs = [], [], []
softmax = nn.Softmax(dim=1)

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs   = softmax(outputs).cpu().numpy()
        preds   = outputs.argmax(1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

y_true = np.array(all_labels)
y_pred = np.array(all_preds)
y_prob = np.array(all_probs)

accuracy  = accuracy_score(y_true, y_pred)
macro_f1  = f1_score(y_true, y_pred, average='macro')
y_bin     = label_binarize(y_true, classes=list(range(5)))
macro_auc = roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro')

cm = confusion_matrix(y_true, y_pred)
per_class = {}
for i, name in enumerate(CLASS_NAMES):
    tp = cm[i, i]; fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp; tn = cm.sum() - tp - fn - fp
    per_class[name] = {
        'sensitivity': round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0,
        'specificity': round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0,
    }

print('=' * 55)
print('  EFFICIENTNET-B0 — TEST SET RESULTS')
print('=' * 55)
print(f'  Accuracy  : {accuracy:.4f}')
print(f'  Macro F1  : {macro_f1:.4f}')
print(f'  Macro AUC : {macro_auc:.4f}')
print('=' * 55)
for name, vals in per_class.items():
    print(f'  {name:<15} Sens: {vals["sensitivity"]:.4f}  Spec: {vals["specificity"]:.4f}')
print('=' * 55)

In [ ]:
# Confusion matrix
cm_norm = confusion_matrix(y_true, y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('EfficientNet-B0 — Normalised Confusion Matrix', fontsize=13)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
save_path = FIGURES_DIR / '04_efficientnet_confusion_matrix.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {save_path}')

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.tab10.colors
for i, name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], linewidth=2, label=f'{name} (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.02])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('EfficientNet-B0 — Per-Class ROC Curves')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_path = FIGURES_DIR / '04_efficientnet_roc_curves.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {save_path}')

In [ ]:
# Overfitting analysis
final_train_acc = history['train_acc'][-1]
gap = final_train_acc - accuracy
print('=' * 55)
print('  EFFICIENTNET-B0 — OVERFITTING ANALYSIS')
print('=' * 55)
print(f'  Final Train Accuracy : {final_train_acc:.4f}')
print(f'  Test Accuracy        : {accuracy:.4f}')
print(f'  Gap                  : {gap:.4f}')
print(f'  Status               : {"WARNING — gap > 5%" if gap > 0.05 else "OK"}')
print('=' * 55)

# Save results
results = {
    'model': 'EfficientNet-B0', 'accuracy': round(accuracy, 4),
    'macro_f1': round(macro_f1, 4), 'macro_auc': round(macro_auc, 4),
    'train_acc': round(final_train_acc, 4), 'gap': round(gap, 4),
    'per_class': per_class,
}
with open(BASE_DIR / 'models' / 'efficientnet_b0_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Results saved.')

### Observation — EfficientNet-B0 Results

EfficientNet-B0 benefits directly from ImageNet pretraining, which gives it low-level feature detectors for edges, textures, and shapes that transfer well to histopathology images. The two-phase strategy prevents catastrophic forgetting during fine-tuning by first adapting only the new head, then gently adjusting the entire network. The resulting accuracy, F1, and AUC scores are compared against the baseline CNN and ResNet-50 in the final results table. Any class-level confusion visible in the matrix will be revisited in the Grad-CAM notebook to understand what the model attends to for difficult classes.

In [ ]:
import shutil
try:
    shutil.copy('/content/04_efficientnet_b0.ipynb',
                str(BASE_DIR / 'notebooks' / '04_efficientnet_b0.ipynb'))
    print('Notebook saved to Drive.')
except:
    print('Use File > Save a copy in Drive to save this notebook.')